In [1]:
import json
import subprocess
from datetime import datetime

with open("query_result.json", "r") as f:
    data = json.load(f)


In [2]:
data

{'numberOfRecords': 151,
 'totalNumberOfRecords': 151,
 'records': [{'flare_class': 'M6.6',
   'start_time': '2011-02-13 17:28:00',
   'peak_time': '2011-02-13 17:38:00',
   'end_time': '2011-02-13 17:47:00',
   'active_region_number': 11158,
   'harp_number': 377},
  {'flare_class': 'X2.2',
   'start_time': '2011-02-15 01:44:00',
   'peak_time': '2011-02-15 01:56:00',
   'end_time': '2011-02-15 02:06:00',
   'active_region_number': 11158,
   'harp_number': 377},
  {'flare_class': 'X1.5',
   'start_time': '2011-03-09 23:13:00',
   'peak_time': '2011-03-09 23:23:00',
   'end_time': '2011-03-09 23:29:00',
   'active_region_number': 11166,
   'harp_number': 401},
  {'flare_class': 'M9.3',
   'start_time': '2011-07-30 02:04:00',
   'peak_time': '2011-07-30 02:09:00',
   'end_time': '2011-07-30 02:12:00',
   'active_region_number': 11261,
   'harp_number': 750},
  {'flare_class': 'M6.0',
   'start_time': '2011-08-03 13:17:00',
   'peak_time': '2011-08-03 13:48:00',
   'end_time': '2011-08-0

In [3]:
records = data['records']

m_flares = [
    flare for flare in records if flare["flare_class"].startswith('M')
]

x_flares = [
    flare for flare in records if flare["flare_class"].startswith('X')
]
selected_flares = m_flares[:15] + x_flares[:15]

In [4]:
print(f"M-class available: {len(m_flares)}")
print(f"X-class available: {len(x_flares)}")
print(f"Selected: {len(selected_flares)}")

M-class available: 94
X-class available: 57
Selected: 30


In [5]:
dates = sorted({
    datetime.strptime(
        flare["peak_time"],
        "%Y-%m-%d %H:%M:%S"
    ).strftime("%m/%d/%Y")
    for flare in selected_flares
})

In [6]:
dates

['01/07/2014',
 '01/23/2012',
 '02/13/2011',
 '02/15/2011',
 '03/07/2012',
 '03/09/2011',
 '03/09/2012',
 '03/10/2012',
 '03/29/2014',
 '04/11/2013',
 '04/25/2014',
 '05/10/2012',
 '07/02/2012',
 '07/04/2012',
 '07/05/2012',
 '07/12/2012',
 '07/30/2011',
 '08/03/2011',
 '08/04/2011',
 '09/06/2011',
 '09/07/2011',
 '09/08/2011',
 '09/10/2014',
 '09/25/2011',
 '10/22/2014',
 '10/24/2014',
 '11/08/2013',
 '11/10/2013']

In [7]:
# for date in dates:
#     cmd = [
#         "globalDownload.py",
#         "--verbose",
#         "--url=https://cedar.openmadrigal.org/",
#         "--outputDir=./tec_hdf5",
#         '--user_fullname=Amrit+Madabushi',
#         "--user_email=am2699@njit.edu",
#         '--user_affiliation=NJIT',
#         '--format=hdf5',
#         f"--startDate={date}",
#         f"--endDate={date}",
#         "--inst=8000",
#         "--kindat=3500"
#     ]
#     print(f"Downloading TEC for {date}")
#     subprocess.run(cmd, check=True)

In [8]:
import h5py
import pandas as pd
import numpy as np
from pathlib import Path

TEC_DIR = Path("tec_hdf5")
CHUNK_SIZE = 250_000

def get_tec_file(flare):
    """Find the daily TEC file corresponding to a flare's peak date."""
    peak = pd.to_datetime(flare["peak_time"])
    date_code = peak.strftime("%y%m%d")

    # print(date_code)
    matches = sorted(TEC_DIR.glob(f"gps{date_code}g.*.hdf5"))

    if not matches:
        raise FileNotFoundError(
            f"No TEC file found for {peak.date()}"
        )
    return matches[0]

def extract_nearest_tec_map(target_time):
    """
    Find the TEC snapshot nearest to target_time wtihout loading the entire daily HDF5 into memory.
    """
    target_time = pd.to_datetime(target_time)

    #Select HDF5 file based on target_date
    date_code = target_time.strftime("%y%m%d")
    matches = sorted(TEC_DIR.glob(f"gps{date_code}g.*.hdf5"))
    if not matches:
        raise FileNotFoundError(
            f"No TEC file found for {target_time.date()}"
        )
    path = matches[0]

    nearest_time = None
    nearest_diff = None

    # PASS 1: find nearest avaliable TEC timestamp
    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0,len(table),CHUNK_SIZE):
            chunk = table[start:start+CHUNK_SIZE]

            times = pd.to_datetime({
                "year": chunk["year"],
                "month": chunk["month"],
                "day": chunk["day"],
                "hour": chunk["hour"],
                "minute": chunk["min"],
                "second": chunk["sec"],
            })

            unique_times = pd.Index(times.unique())

            diffs = abs(unique_times-target_time)
            i = diffs.argmin()

            candidate_time = unique_times[i]
            candidate_diff = diffs[i]

            if nearest_diff is None or candidate_diff < nearest_diff:
                nearest_diff = candidate_diff
                nearest_time = candidate_time
        print(f"Requested: {target_time}")
    print(f"Nearest TEC map: {nearest_time}")
    print(f"Difference: {nearest_diff}")
    if nearest_diff > pd.Timedelta(minutes=5):
        raise ValueError(
            f"TEC timestamp mismatch: requested {target_time}, nearest {nearest_time}, difference {nearest_diff}"
        )

    # PASS 2: extract only rows belonging to that TEC timestamp
    pieces = []

    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0, len(table), CHUNK_SIZE):
            chunk = table[start:start + CHUNK_SIZE]

            mask = (
                (chunk["year"] == nearest_time.year) &
                (chunk["month"] == nearest_time.month) &
                (chunk["day"] == nearest_time.day) &
                (chunk["hour"] == nearest_time.hour) &
                (chunk["min"] == nearest_time.minute) &
                (chunk["sec"] == nearest_time.second)
            )

            if np.any(mask):
                selected = chunk[mask]

                pieces.append(
                    pd.DataFrame({
                        "gdlat": selected["gdlat"],
                        "glon": selected["glon"],
                        "tec": selected["tec"],
                    })
                )

    if not pieces:
        raise RuntimeError(
            f"No TEC rows found for {nearest_time}"
        )

    tec_map = pd.concat(pieces, ignore_index=True)

    # Madrigal missing-value convention
    tec_map.loc[tec_map["tec"] > 1e20, "tec"] = np.nan

    return nearest_time, tec_map

In [9]:
flare = selected_flares[0]

path = get_tec_file(flare)

print("Flare:", flare["flare_class"], flare["peak_time"])
print("TEC file:", path.name)

tec_time, tec_map = extract_nearest_tec_map(
    flare["peak_time"]
)

print(tec_map.shape)
print(tec_map.head())

Flare: M6.6 2011-02-13 17:38:00
TEC file: gps110213g.002.hdf5
Requested: 2011-02-13 17:38:00
Nearest TEC map: 2011-02-13 17:37:30
Difference: 0 days 00:00:30
(6687, 3)
   gdlat  glon   tec
0  -90.0 -84.0  66.0
1  -90.0 -82.0  65.9
2  -90.0 -81.0  65.9
3  -90.0 -80.0  65.8
4  -90.0 -79.0  65.7


In [10]:
from pathlib import Path

print("cwd:", Path.cwd())
print("TEC_DIR:", TEC_DIR.resolve())
print("exists:", TEC_DIR.exists())
print("files found:", list(TEC_DIR.glob("gps110213g*.hdf5")))

cwd: /home/amritm/projects/surya-tec-forecasting
TEC_DIR: /home/amritm/projects/surya-tec-forecasting/tec_hdf5
exists: True
files found: [PosixPath('tec_hdf5/gps110213g.002.hdf5')]


# Visualization

In [11]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

FIGURE_DIR = Path("figures/tec_maps")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def plot_map(tec_map, timestamp):
    """Plot one global TEC map on the given axes."""
    grid = tec_map.pivot(
        index="gdlat",
        columns="glon",
        values="tec"
    )

    fig,ax = plt.subplots(figsize=(12,5))

    mesh = ax.pcolormesh(
        grid.columns,
        grid.index,
        grid.values,
        shading="auto"
    )

    fig.colorbar(mesh,ax=ax,label="TEC")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Global TEC Map —  {timestamp}")

    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)

    timestamp = pd.to_datetime(timestamp)
    filename = f"tec_{timestamp.strftime('%Y%m%d_%H%M%S')}.png"
    save_path = FIGURE_DIR / filename

    fig.savefig(save_path, dpi=300, bbox_inches="tight")

    print(f"Saved: {save_path}")

    # plt.show()
    plt.close(fig)

    # lats = sorted(map_df["gdlat"].unique())
    # lons = sorted(map_df["glon"].unique())
    # tec_grid = map_df.pivot(index="gdlat", columns="glon", values="tec")

    # # pcolormesh needs cell edges, not centers — shift by 0.5°
    # lat_edges = [l - 0.5 for l in lats] + [lats[-1] + 0.5]
    # lon_edges = [l - 0.5 for l in lons] + [lons[-1] + 0.5]
    # LON, LAT = np.meshgrid(lon_edges, lat_edges)

    # tec_arr = tec_grid.reindex(index=lats, columns=lons).values

    # im = ax.pcolormesh(LON, LAT, tec_arr,
    #                    cmap="viridis", vmin=0, vmax=60,
    #                    shading="flat")
    # ax.set_xlim(-180, 180)
    # ax.set_ylim(-90, 90)
    # ax.set_title(title, fontsize=9, pad=4)
    # ax.set_aspect("equal")
    # ax.tick_params(labelsize=7)
    # # Draw equator and prime meridian for reference
    # ax.axhline(0, color="white", linewidth=0.4, alpha=0.5)
    # ax.axvline(0, color="white", linewidth=0.4, alpha=0.5)
    # return im

In [12]:
# plot_map(tec_map,tec_time)

In [13]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])

#     for offset in [-3,-2, -1, 0, 1, 2, 3]:
#         target_time = peak + pd.Timedelta(hours=offset)

#         tec_time, tec_map = extract_nearest_tec_map(target_time)

#         plot_map(tec_map, tec_time)

## Covering missing dates

In [14]:
required_dates = {
    (pd.to_datetime(flare["peak_time"]) + pd.Timedelta(hours = offset)).date()
    for flare in selected_flares
    for offset in range(-3,4)
}

available_dates = {
    pd.to_datetime(path.name[3:9], format="%y%m%d").date()
    for path in TEC_DIR.glob("gps*g.*.hdf5")
}

missing_dates = sorted(required_dates-available_dates)

print("Required dates:", len(required_dates))
print("Available dates:", len(available_dates))
print("Missing dates:", missing_dates)

Required dates: 35
Available dates: 35
Missing dates: []


In [15]:
# dates = [
#     date.strftime("%m/%d/%Y")
#     for date in missing_dates
# ]

# for date in dates:
#     cmd = [
#         "globalDownload.py",
#         "--verbose",
#         "--url=https://cedar.openmadrigal.org/",
#         "--outputDir=./tec_hdf5",
#         "--user_fullname=Amrit+Madabushi",
#         "--user_email=am2699@njit.edu",
#         "--user_affiliation=NJIT",
#         "--format=hdf5",
#         f"--startDate={date}",
#         f"--endDate={date}",
#         "--inst=8000",
#         "--kindat=3500"
#     ]

#     print(f"Downloading TEC for {date}")
#     subprocess.run(cmd, check=True)

In [16]:
from pathlib import Path
import pandas as pd

TEC_DIR = Path("tec_hdf5")

extra_files = []

for path in TEC_DIR.glob("*.hdf5"):
    file_date = pd.to_datetime(
        path.name[3:9], format="%y%m%d"
    ).date()

    if file_date not in required_dates or "_0.hdf5" in path.name:
        extra_files.append(path)

print("Extra files:")
for path in extra_files:
    print(path.name)

Extra files:


## Obtain T-4 solar observations from Amazon S3 Nasa Surya Bench

In [17]:
import boto3
from botocore import UNSIGNED

In [18]:
config = boto3.session.Config(signature_version=UNSIGNED)
s3 = boto3.client("s3", config=config)
bucket = 'nasa-surya-bench'

In [19]:
# response = s3.list_objects_v2(Bucket=bucket)
# for obj in response.get('Contents', []):
#     print(obj['Key'])

In [20]:
dates

['01/07/2014',
 '01/23/2012',
 '02/13/2011',
 '02/15/2011',
 '03/07/2012',
 '03/09/2011',
 '03/09/2012',
 '03/10/2012',
 '03/29/2014',
 '04/11/2013',
 '04/25/2014',
 '05/10/2012',
 '07/02/2012',
 '07/04/2012',
 '07/05/2012',
 '07/12/2012',
 '07/30/2011',
 '08/03/2011',
 '08/04/2011',
 '09/06/2011',
 '09/07/2011',
 '09/08/2011',
 '09/10/2014',
 '09/25/2011',
 '10/22/2014',
 '10/24/2014',
 '11/08/2013',
 '11/10/2013']

In [21]:
from datetime import datetime

# for date_str in dates:
#     date = datetime.strptime(date_str, "%m/%d/%Y")
#     prefix = date.strftime("%Y/%m/%Y%m%d_")

#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     print(date_str, len(response.get("Contents", [])))

In [22]:
# for flare in selected_flares:
#     print(flare["peak_time"])

In [23]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=4)
#     print(f"Peak: {peak} Cutoff: {cutoff}")

In [24]:
# responses = []
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=4)
#     prefix = cutoff.strftime("%Y/%m/%Y%m%d_")
#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     keys = [obj["Key"] for obj in response.get("Contents", [])]

#     valid_keys = [
#         key for key in keys
#         if pd.to_datetime(key.split("/")[-1], format="%Y%m%d_%H%M.nc") <= cutoff
#     ]

#     if not valid_keys:
#         print(f"No valid observation found for {peak}")
#         continue

#     selected_key = max(valid_keys)

#     responses.append((peak,cutoff,selected_key))

#     print(date_str, len(response.get("Contents", [])))
#     print(f"Peak: {peak} | Cutoff: {cutoff} | S3: {selected_key}")

Note: September 10, 2014 measurement has a temporal difference of 45 mins between the selected flare observation and cutoff

In [25]:
# from pathlib import Path

# output_dir = Path("solar_observations")
# output_dir.mkdir(exist_ok=True)

In [26]:
# for peak,cutoff,key in responses:
#     destination = output_dir / Path(key).name

#     if not destination.exists(): # avoid redownloading
#         s3.download_file(bucket, key, str(destination))

#     print(f"Downloaded: {destination}")

In [27]:
# from pathlib import Path

# solar_files = sorted(Path("solar_observations").glob("*.nc"))

# print(f"Total solar files: {len(solar_files)}")

# for file in solar_files:
#     print(file.name)

In [28]:
# for flare in selected_flares:    
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak-pd.Timedelta(hours=5)
#     prefix = cutoff.strftime("%Y/%m/%Y%m%d_")
#     response = s3.list_objects_v2(
#         Bucket=bucket,
#         Prefix=prefix
#     )

#     keys = [obj["Key"] for obj in response.get("Contents", [])]

#     valid_keys = [
#         key for key in keys
#         if pd.to_datetime(key.split("/")[-1], format="%Y%m%d_%H%M.nc") <= cutoff
#     ]

#     if not valid_keys:
#         print(f"No valid observation found for {peak}")
#         continue

#     selected_key = max(valid_keys)
#     destination = output_dir / Path(selected_key).name
#     if not destination.exists():
#         s3.download_file(bucket, selected_key, str(destination))
#         print(f"Downloaded: {destination.name}")


In [29]:
# from pathlib import Path

# solar_files = list(Path("solar_observations").glob("*.nc"))

# print(f"Total files: {len(solar_files)}")
# print(f"Total size: {sum(f.stat().st_size for f in solar_files) / 1024**3:.2f} GiB")

In [30]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff = peak - pd.Timedelta(hours=5)

#     candidates = []

#     for path in output_dir.glob("*.nc"):
#         observation_time = pd.to_datetime(
#             path.stem,
#             format="%Y%m%d_%H%M"
#         )

#         if observation_time <= cutoff:
#             candidates.append((observation_time, path.name))

#     if candidates:
#         observation_time, filename = max(candidates)
#         difference = (cutoff - observation_time).total_seconds() / 60

#         print(
#             f"Peak: {peak} | "
#             f"T-5 Cutoff: {cutoff} | "
#             f"Selected: {observation_time} | "
#             f"Difference: {difference:.0f} min | "
#             f"File: {filename}"
#         )

In [31]:
# for flare in selected_flares:
#     peak = pd.to_datetime(flare["peak_time"])
#     cutoff_t4 = peak - pd.Timedelta(hours=4)
#     cutoff_t5 = peak - pd.Timedelta(hours=5)

#     candidates_t4 = []
#     candidates_t5 = []

#     for path in output_dir.glob("*.nc"):
#         observation_time = pd.to_datetime(path.stem, format="%Y%m%d_%H%M")

#         if observation_time <= cutoff_t4:
#             candidates_t4.append((observation_time, path.name))

#         if observation_time <= cutoff_t5:
#             candidates_t5.append((observation_time, path.name))

#     if candidates_t4 and candidates_t5:
#         t4, file_t4 = max(candidates_t4)
#         t5, file_t5 = max(candidates_t5)

#         gap = (t4 - t5).total_seconds() / 60

#         print(
#             f"Peak: {peak} | "
#             f"T-5: {t5} | "
#             f"T-4: {t4} | "
#             f"Gap: {gap:.0f} min"
#         )

The gap between $t-5$ and $t-4$ for the September 10, 2014 flare is only 24 minutes, consistent with the previously identified 45-minute offset at $t-4$. The March 9th, 2011 flare a gap of 48 minutes. All other solar observations have a 60 minute gap between $t-4$ and $t-5$

## Training

### 0. Preprocessing

In [32]:
# print(selected_flares)

In [33]:
# tec_files = list(Path("figures/tec_maps").glob("tec_*.png"))

In [34]:
# flare_df = pd.DataFrame(selected_flares)
# for offset in [-5, -4]:
#     flare_df[f"solar_t{offset}"] = None

# for offset in range(-3, 4):
#     flare_df[f"tec_t{offset:+d}"] = None

# solar_times = {
#     path: pd.to_datetime(path.stem,format="%Y%m%d_%H%M")
#     for path in solar_files
# }

# tec_times = {
#     path: pd.to_datetime(
#         path.stem.removeprefix("tec_"),
#         format="%Y%m%d_%H%M%S"
#     )
#     for path in tec_files
# }
# hdf5_files = list(Path("tec_hdf5").glob("*.hdf5"))
# hdf5_by_date = {
#     pd.to_datetime(path.name[3:9], format="%y%m%d").date(): path
#     for path in hdf5_files
# }

In [35]:
# with h5py.File("tec_hdf5/gps110213g.002.hdf5", "r") as f:
#     dataset = f["Data"]["Table Layout"]

#     print(dataset.shape)
#     print(dataset.dtype.names)

In [36]:
# for idx, flare in flare_df.iterrows():
#     peak_time = pd.Timestamp(flare["peak_time"])

#     # Match existing solar observations to solar_t5 and solar_t4
#     for offset in [-5,-4]:
#         target_time = peak_time + pd.Timedelta(hours=offset)
        
#         closest_file = min(
#             solar_times,
#             key=lambda path: abs(solar_times[path] - target_time)
#         )

#         flare_df.at[idx, f"solar_t{offset:+d}"] = str(closest_file)
            
#     # Match existing TEC maps to the seven target timestamps
#     for offset in range(-3, 4):
#         col = f"tec_t{offset:+d}"
#         target_time = peak_time + pd.Timedelta(hours=offset)

#         closest_file = min(
#             tec_times,
#             key=lambda path: abs(tec_times[path] - target_time)
#         )

#         flare_df.at[idx, col] = str(closest_file)

#         tec_time = tec_times[closest_file]

#         flare_df.at[idx, f"{col}_hdf5"] = str(
#             hdf5_by_date.get(tec_time.date(), "")
#         )
            
# flare_df.to_csv("flare_manifest.csv", index=False)

In [37]:
def grid_tec(tec_map, resolution=1.0):
    '''
    This function is used to create a grid based on latitude, longitude and total electron content (TEC) measurements 
    provided by the map.
    
    Longitude coordinates are normalized to [-180, 180], and spatial binning is applied to latitude and longitude. 
    TEC measurements within each grid cell are averaged, with cells containing no valid observations reported as NaN. 
    '''
    lat = tec_map["gdlat"].to_numpy()
    lon = tec_map["glon"].to_numpy()
    tec = tec_map["tec"].to_numpy()

    # Normalize Longitude
    lon = ((lon+180)%360)-180

    # Remove invalid measurements
    valid = (np.isfinite(lat) & np.isfinite(lon) & np.isfinite(tec))

    lat = lat[valid]
    lon = lon[valid]
    tec = tec[valid]

    # Grid boundaries
    if resolution <= 0 or not np.isclose(180 / resolution, round(180 / resolution)):
        raise ValueError("Resolution must evenly divide 180 degrees.")
    lat_edges = np.linspace(-90,90,int(180/resolution)+1)
    lon_edges = np.linspace(-180,180,int(360/resolution)+1)

    # Aggregate TEC values within each grid cell
    tec_sum, _, _ = np.histogram2d(
        lat, lon,
        bins=[lat_edges, lon_edges],
        weights=tec
    )

    # Count meaasurements within each cell
    counts, _, _ = np.histogram2d(
        lat, lon,
        bins=[lat_edges, lon_edges]
    )

    # Calculate mean TEC per cell
    grid = np.full(tec_sum.shape, np.nan, dtype=np.float32)

    np.divide(tec_sum,counts,out=grid,where=counts>0)

    return grid

In [38]:
def extract_nearest_tec_map(target_time, path):
    """
    Extract the TEC map from the hdf5 file
    """
    target_time = pd.to_datetime(target_time)


    nearest_time = None
    nearest_diff = None

    # PASS 1: find nearest avaliable TEC timestamp
    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0,len(table),CHUNK_SIZE):
            chunk = table[start:start+CHUNK_SIZE]

            times = pd.to_datetime({
                "year": chunk["year"],
                "month": chunk["month"],
                "day": chunk["day"],
                "hour": chunk["hour"],
                "minute": chunk["min"],
                "second": chunk["sec"],
            })

            unique_times = pd.Index(times.unique())

            diffs = abs(unique_times-target_time)
            i = diffs.argmin()

            candidate_time = unique_times[i]
            candidate_diff = diffs[i]

            if nearest_diff is None or candidate_diff < nearest_diff:
                nearest_diff = candidate_diff
                nearest_time = candidate_time
        print(f"Requested: {target_time}")
    print(f"Nearest TEC map: {nearest_time}")
    print(f"Difference: {nearest_diff}")
    if nearest_diff > pd.Timedelta(minutes=5):
        raise ValueError(
            f"TEC timestamp mismatch: requested {target_time}, nearest {nearest_time}, difference {nearest_diff}"
        )

    # PASS 2: extract only rows belonging to that TEC timestamp
    pieces = []

    with h5py.File(path, "r") as f:
        table = f["Data"]["Table Layout"]

        for start in range(0, len(table), CHUNK_SIZE):
            chunk = table[start:start + CHUNK_SIZE]

            mask = (
                (chunk["year"] == nearest_time.year) &
                (chunk["month"] == nearest_time.month) &
                (chunk["day"] == nearest_time.day) &
                (chunk["hour"] == nearest_time.hour) &
                (chunk["min"] == nearest_time.minute) &
                (chunk["sec"] == nearest_time.second)
            )

            if np.any(mask):
                selected = chunk[mask]

                pieces.append(
                    pd.DataFrame({
                        "gdlat": selected["gdlat"],
                        "glon": selected["glon"],
                        "tec": selected["tec"],
                    })
                )

    if not pieces:
        raise RuntimeError(
            f"No TEC rows found for {nearest_time}"
        )

    tec_map = pd.concat(pieces, ignore_index=True)

    # Madrigal missing-value convention
    tec_map.loc[tec_map["tec"] > 1e20, "tec"] = np.nan

    return nearest_time, tec_map

In [39]:
# Used to generate the precomputed TEC Grids stored in npz files instead of accessing from hdf5 files
# from pathlib import Path

# output_dir = Path("tec_grids")
# output_dir.mkdir(exist_ok=True)

# for idx,flare in dataset.df.iterrows():
#     tec, tec_mask = dataset.load_tec(flare)

#     output_path = output_dir / f"flare_{idx:03d}.npz"

#     np.savez_compressed(
#         output_path,
#         tec=tec,
#         tec_mask = tec_mask,
#         resolution=1.0
#     )
#     dataset.df.loc[idx, "tec_grid_path"] = str(output_path)


# dataset.df.to_csv("flare_manifest.csv", index=False)

In [40]:
# target_time = pd.Timestamp("2011-02-13 14:38:00")

# tec_time, tec_map = extract_nearest_tec_map(target_time)

# tec_grid = grid_tec(tec_map)

# print("Timestamp:", tec_time)
# print("Grid shape:", tec_grid.shape)
# print("Valid cells:", np.isfinite(tec_grid).sum())
# print("Missing cells:", np.isnan(tec_grid).sum())

In [41]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12, 6))

# plt.imshow(
#     tec_grid,
#     origin="lower",
#     extent=[-180, 180, -90, 90],
#     aspect="auto",
#     cmap="viridis"
# )

# plt.colorbar(label="TEC (TECU)")
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.title(f"Gridded TEC — {tec_time}")

# plt.show()

In [42]:
# for resolution in [0.5, 1.0, 2.0]:
#     grid = grid_tec(tec_map, resolution=resolution)

#     coverage = np.isfinite(grid).mean() * 100

#     print(
#         f"Resolution: {resolution}° | "
#         f"Shape: {grid.shape} | "
#         f"Coverage: {coverage:.2f}%"
#     )

Load scalers from Surya (Normalize into shape $X\in{R^{13*2*4096*4096}}$ )

In [43]:
import pandas as pd
from torch.utils.data import Dataset
import yaml
from surya.utils.data import build_scalers
from surya.datasets.helio import transform
import numpy as np
import xarray as xr
import torch

class FlareDataset(Dataset):
    def __init__(self,manifest_path):
        self.df = pd.read_csv(manifest_path)
        with open("../Surya/data/Surya-1.0/scalers.yaml") as f:
            scaler_info = yaml.safe_load(f)

        scalers = build_scalers(scaler_info)
        self.channels = [
            "aia94", "aia131", "aia171", "aia193",
            "aia211", "aia304", "aia335", "aia1600",
            "hmi_m", "hmi_bx", "hmi_by", "hmi_bz", "hmi_v"
        ]
        self.means = np.array([scalers[ch].mean for ch in self.channels])
        self.stds = np.array([scalers[ch].std for ch in self.channels])
        self.scale_factors = np.array([scalers[ch].sl_scale_factor for ch in self.channels])
        self.epsilons = np.array([scalers[ch].epsilon for ch in self.channels])


    def __len__(self):
        return len(self.df)
    def load_solar(self, path):
        with xr.open_dataset(
            path,
            engine="h5netcdf",
            chunks=None,
            cache=False
        ) as ds:
            solar = ds[self.channels].to_array().load().to_numpy()

        return transform(
            solar,
            self.means,
            self.stds,
            self.scale_factors,
            self.epsilons
        )
    def load_tec(self, path):        
        with np.load(path) as data:
            tec = data["tec"]
            tec_mask = data["tec_mask"]

        return tec, tec_mask
    
        # OLD
        # peak_time = pd.Timestamp(flare["peak_time"])
        # tec_grids = []

        # for offset in range(-3,4):
        #     label = f"tec_t{offset:+d}"

        #     path = flare[f"{label}_hdf5"]
        #     target_time = peak_time + pd.Timedelta(hours=offset)

        #     _,tec_map = extract_nearest_tec_map(target_time, path)

        #     tec_grid = grid_tec(tec_map, resolution=1.0)
        #     tec_grids.append(tec_grid)

        # tec = np.stack(tec_grids,axis=0)

        # tec_mask = np.isfinite(tec)
        # tec = np.nan_to_num(tec,nan=0.0).astype(np.float32)

        # return tec,tec_mask


    def __getitem__(self, idx):
        flare = self.df.iloc[idx]

        solar_t5 = self.load_solar(flare["solar_t-5"])
        solar_t4 = self.load_solar(flare["solar_t-4"])

        solar = np.stack([solar_t5, solar_t4], axis=1)
        tec, tec_mask = self.load_tec(flare["tec_grid_path"])

        return {
            "solar": torch.from_numpy(solar),
            "tec": torch.from_numpy(tec),
            "tec_mask": torch.from_numpy(tec_mask),
            "flare_class": flare["flare_class"],
            "peak_time": flare["peak_time"]
        }
    def get_flares_by_class(self, flare_class):
        return self.df[self.df["flare_class"] == flare_class]

In [44]:
dataset = FlareDataset("flare_manifest.csv")


In [45]:
print(len(dataset))

sample = dataset[0]

print(sample["solar"].shape)
print(sample["solar"].dtype)
print(torch.isfinite(sample["solar"]).all())

30
torch.Size([13, 2, 4096, 4096])
torch.float32
tensor(True)


In [46]:
flare15 = dataset.__getitem__(idx=15)
flare15

{'solar': tensor([[[[-2.7955e-01, -4.5286e-01, -4.7089e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-3.6507e-01, -3.1025e-01, -4.9601e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-3.9150e-01, -2.9382e-01, -4.9627e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           ...,
           [-4.5121e-01, -4.4202e-01, -3.7811e-01,  ..., -2.0305e-01,
            -2.0305e-01, -5.4036e-01],
           [-3.8197e-01, -3.4819e-01, -3.7878e-01,  ..., -1.1571e-01,
            -1.7886e-01, -5.4036e-01],
           [-5.4036e-01, -1.4429e-01, -4.5878e-01,  ..., -1.1571e-01,
            -1.7886e-01, -5.4036e-01]],
 
          [[-4.4656e-01, -2.9547e-01, -2.9696e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-4.9893e-01, -5.4036e-01, -5.4036e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.4036e-01],
           [-4.9165e-01, -5.4036e-01, -4.2959e-01,  ..., -5.4036e-01,
            -5.4036e-01, -5.40

In [47]:
# Inspect Flares according to class
dataset.get_flares_by_class("M9.3")

,flare_class,start_time,peak_time,end_time,active_region_number,harp_number,solar_t-5,solar_t-4,tec_t-3,tec_t-2,...,tec_t+2,tec_t+3,tec_t-3_hdf5,tec_t-2_hdf5,tec_t-1_hdf5,tec_t+0_hdf5,tec_t+1_hdf5,tec_t+2_hdf5,tec_t+3_hdf5,tec_grid_path
1,M9.3,2011-07-30 02:04:00,2011-07-30 02:09:00,2011-07-30 02:12:00,11261,750,solar_observations/20110729_2100.nc,solar_observations/20110729_2200.nc,figures/tec_maps/tec_20110729_230730.png,figures/tec_maps/tec_20110730_000730.png,...,figures/tec_maps/tec_20110730_040730.png,figures/tec_maps/tec_20110730_050730.png,tec_hdf5/gps110729g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_hdf5/gps110730g.001.hdf5,tec_grids/flare_001.npz
3,M9.3,2011-08-04 03:41:00,2011-08-04 03:57:00,2011-08-04 04:04:00,11261,750,solar_observations/20110803_2248.nc,solar_observations/20110803_2348.nc,figures/tec_maps/tec_20110804_005730.png,figures/tec_maps/tec_20110804_015730.png,...,figures/tec_maps/tec_20110804_055730.png,figures/tec_maps/tec_20110804_065730.png,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_hdf5/gps110804g.002.hdf5,tec_grids/flare_003.npz


$TEC_{i,j} = \frac{1}{N_{i,j}}\sum_{k\in{B_{i,j}}}{TEC_k}$

For this experiment, we will be using 1.0° resolution.

### 1. Loading LoRA Trained Surya

In [48]:
from surya.models.helio_spectformer import HelioSpectFormer
from downstream_examples.solar_flare_forcasting.models import HelioSpectformer2D

/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/amritm/projects/surya-tec-forecasting/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [49]:
import yaml

with open ("../Surya/downstream_examples/solar_flare_forcasting/config.yaml","r") as f:
    config = yaml.safe_load(f)


In [50]:
config["model"]["ft_unembedding_type"] = "linear"
config["model"]["ft_out_chans"] = 7

Instead of using Surya's in-built Linear Decoder, our TEC Prediction head will use a custom configuration.

In [51]:
# print(config["model"]["model_type"])
# print(config["model"]["embed_dim"])
# print(config["model"]["time_embedding"])

In [52]:
import torch

model = HelioSpectformer2D(
    img_size=config["model"]["img_size"],
    patch_size=config["model"]["patch_size"],
    in_chans=config["model"]["in_channels"],
    embed_dim=config["model"]["embed_dim"],
    time_embedding=config["model"]["time_embedding"],
    depth=config["model"]["depth"],
    n_spectral_blocks=config["model"]["spectral_blocks"],
    num_heads=config["model"]["num_heads"],
    mlp_ratio=config["model"]["mlp_ratio"],
    drop_rate=config["model"]["drop_rate"],
    window_size=config["model"]["window_size"],
    dp_rank=config["model"]["dp_rank"],
    learned_flow=config["model"]["learned_flow"],
    use_latitude_in_learned_flow=config["use_latitude_in_learned_flow"],
    init_weights=config["model"]["init_weights"],
    checkpoint_layers=config["model"]["checkpoint_layers"],
    rpe=config["model"]["rpe"],
    finetune=True,
    dtype=torch.float32,
    config=config,
)

In [53]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

Parameters: 364.22M


In [54]:
checkpoint = torch.load(
    "../Surya/data/Surya-1.0/surya.366m.v1.pt",
    map_location="cpu",
    weights_only=True
)

model_state = model.state_dict()

filtered_checkpoint = {
            k: v
            for k, v in checkpoint.items()
            if k in model_state and v.shape == model_state[k].shape
}
print(f"Matched: {len(filtered_checkpoint)} / {len(checkpoint)} tensors")

Matched: 157 / 159 tensors


In [55]:
for k, v in checkpoint.items():
    if k not in filtered_checkpoint:
        print(k, v.shape)

unembed.unembed.0.weight torch.Size([3328, 1280, 1, 1])
unembed.unembed.0.bias torch.Size([3328])


In [56]:
# model_state.update(filtered_checkpoint)
model.load_state_dict(filtered_checkpoint, strict=False)

print(torch.equal(
    model.state_dict()["embedding.patch_embed.proj.weight"],
    checkpoint["embedding.patch_embed.proj.weight"]
))

True


In [57]:
for name,module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if "attn" in name:
            print(name, module.weight.shape)

backbone.blocks_attention.0.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.0.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.0.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.1.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.1.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.1.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.2.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.2.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.2.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.3.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.3.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.3.attn.to_dynamic_projection torch.Size([64, 1280])
backbone.blocks_attention.4.attn.qkv torch.Size([3840, 1280])
backbone.blocks_attention.4.attn.proj torch.Size([1280, 1280])
backbone.blocks_attention.4.attn.to_dynamic_projection torch.Si

In [58]:
from peft import LoraConfig, get_peft_model

In [59]:
targets = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]

for target in targets:
    matches = [
        name for name, module in model.named_modules()
        if name.endswith(target) and isinstance(module, torch.nn.Linear)
    ]
    print(f"{target}: {len(matches)} matches")

q_proj: 0 matches
v_proj: 0 matches
k_proj: 0 matches
out_proj: 0 matches
fc1: 10 matches
fc2: 10 matches


```
# Note: We will first test Surya LoRA Config using two out of three linear attention projections (query,key,value and output projections) before testing dynamic projection. 
```

In [60]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")

In [61]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["attn.qkv", "attn.proj"],
    lora_dropout=.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model = model.to(device)

In [62]:
model.print_trainable_parameters()

trainable params: 491,520 || all params: 364,713,984 || trainable%: 0.1348


In [63]:
for name,param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

base_model.model.backbone.blocks_attention.0.attn.qkv.lora_A.default.weight torch.Size([8, 1280])
base_model.model.backbone.blocks_attention.0.attn.qkv.lora_B.default.weight torch.Size([3840, 8])
base_model.model.backbone.blocks_attention.0.attn.proj.lora_A.default.weight torch.Size([8, 1280])
base_model.model.backbone.blocks_attention.0.attn.proj.lora_B.default.weight torch.Size([1280, 8])
base_model.model.backbone.blocks_attention.1.attn.qkv.lora_A.default.weight torch.Size([8, 1280])
base_model.model.backbone.blocks_attention.1.attn.qkv.lora_B.default.weight torch.Size([3840, 8])
base_model.model.backbone.blocks_attention.1.attn.proj.lora_A.default.weight torch.Size([8, 1280])
base_model.model.backbone.blocks_attention.1.attn.proj.lora_B.default.weight torch.Size([1280, 8])
base_model.model.backbone.blocks_attention.2.attn.qkv.lora_A.default.weight torch.Size([8, 1280])
base_model.model.backbone.blocks_attention.2.attn.qkv.lora_B.default.weight torch.Size([3840, 8])
base_model.model

In [64]:
print("Patch size:", model.patch_size)
print("Embedding dimension:", model.embed_dim)
print("Global tokens:", model.nglo)
print("Fine-tuning mode:", model.finetune)

Patch size: 16
Embedding dimension: 1280
Global tokens: 0
Fine-tuning mode: True


### 2. Training

Input: $ X \in \mathbb{R}^{13*2*4096*4096}$

Predictions: $\hat{Y} \in \mathbb{R}^{7*180*360}$

$L = (\frac{4096}{16})^2 = 256^2 = 65,356$
, $D = 1280$

$Z_{latent representations} \in{\mathbb{R}^{B*L*1280}}$ = ~84M latent values per observation

$\mathcal{L}_{masked MSE} = \frac{\sum_{t,i,j}{M_{t,i,j}(\hat{Y}_{t,i,j}-Y_{t,i,j})^2}}{\sum_{t,i,j}{M_{t,i,j}}}$

In [65]:
from sklearn.model_selection import LeaveOneOut
from torch.utils.data import Subset,DataLoader
import numpy as np
from surya.models.helio_spectformer import HelioSpectFormer
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

```text
Potential Structure:
Surya transformer
       |
       v
Spatial tokens (B, L, D)
       |
       v
Reshape to (B, D, H', W')
       |
       v
1×1 Conv (reduce feature channels)
       |
       v
Spatial resizing
       |
       v
3×3 Conv + GELU
       |
       v
3×3 Conv (7 output channels)
       |
       v
TEC predictions (B, 7, 180, 360)
```

$X_{solar} = [X_{t-5}, X_{t-4}]$, where each observation has 13 channels.



In [66]:
flare_ids = np.arange(len(dataset))
loo = LeaveOneOut()

for fold, (train_idx, test_idx) in enumerate(loo.split(flare_ids)):
    train_dataset = Subset(dataset, train_idx.tolist())
    test_dataset = Subset(dataset, test_idx.tolist())

    trainloader = DataLoader(
        dataset=train_dataset,
        batch_size=1,
        shuffle=True
    )
    testloader = DataLoader(
        dataset=test_dataset,
        batch_size=1,
        shuffle=False
    )

    # print(f"Fold {fold+1}")
    # print(f"Training flares: {len(train_idx)}")
    # print(f"Testing flare: {test_idx[0]}")

    # held_out = dataset.df.iloc[test_idx[0]]

    # print(f"Held-out flare: {held_out['peak_time']}")
    # print(f"Flare class: {held_out['flare_class']}")

In [67]:
# TODO: CONFIGURE SURYA LORA TRAINED MODEL WITH DATASET
def masked_mse(pred,target,mask):
    squared_error = (pred-target).square()
    return (mask*squared_error).sum()/mask.sum().clamp(min=1)

In [68]:
# flare2 = dataset.__getitem__(idx=2)
# flare2["solar"].shape # torch.Size([13, 2, 4096, 4096])
# solar_input = flare2["solar"].unsqueeze(0)
# print(solar_input.shape)

# batch = {
#     "ts": solar_input.to(device),
#     "time_delta_input" : torch.tensor(
#         [[-1.0, 0.0]],
#         device=device
#     )
# }

Shape: $ [B, C, T, H, W] $ where:


B - Batch,
C - Channel,
T- Time,
H - Height,
W - Width

In [69]:
# model.eval()

# with torch.no_grad():
#     tokens = HelioSpectFormer.forward(model, batch=batch)

# print(f"Latent Representations: {tokens.shape}")

Latent Representation Shape: $[1, 65536, 1280]$

In [70]:
# train_batch = next(iter(trainloader))

# print("Solar:", train_batch["solar"].shape)
# print("TEC:", train_batch["tec"].shape)

```text
Solar: torch.Size([1, 13, 2, 4096, 4096])
TEC: torch.Size([1, 7, 180, 360])
```

In [71]:
### We are saving flares on disk to avoid having Surya recompute the flares every time


# output_dir = Path("latent_features")
# output_dir.mkdir(exist_ok=True)

# model.eval()

# latent_features = []

# with torch.inference_mode():
#     for i in range(len(dataset)):
#         flare = dataset[i]

#         batch = {
#             "ts": flare["solar"].unsqueeze(0).to(device),
#             "time_delta_input" : torch.tensor(
#                 [[-1.0, 0.0]],
#                 device=device
#             )
#         }
#         tokens = HelioSpectFormer.forward(model, batch=batch)

#         torch.save(tokens.cpu(), output_dir / f"flare_{i:02d}.pt")

#         print(f"Saved flare {i + 1}/{len(dataset)}", flush=True)

In [72]:
# pooled_dir = Path("pooled_features")
# pooled_dir.mkdir(exist_ok=True)

# for i in range(len(dataset)):
#     tokens = torch.load(
#         Path("latent_features") / f"flare_{i:02d}.pt",
#         map_location="cpu",
#         weights_only=True
#     )

#     pooled = tokens.mean(dim=1)

#     torch.save(pooled, pooled_dir / f"flare_{i:02d}.pt")

#     print(f"Pooled flare {i + 1}/{len(dataset)}")

In [83]:
class TECLinearHead(nn.Module):
    def __init__(self, 
                input_dim=1280, 
                output_dim=7*18*36,  
                low_res=(36, 72),
                output_res=(180, 360),
                num_frames=7):
        super().__init__()
        self.low_res = low_res
        self.output_res = output_res
        self.num_frames = num_frames

        h,w = low_res

        self.linear = nn.Linear(input_dim, num_frames * h * w)

    def forward(self,x):
        x = self.linear(x)

        # Reshape X to 7 low-resolution TEC Maps
        x = x.reshape(-1,self.num_frames,
                      *self.low_res)

        #Upscale into the original TEC resolution
        if self.low_res != self.output_res:
            x = F.interpolate(
                x,
                size=(180,360),
                mode="bilinear",
                align_corners=False
            )

        return x

In [74]:
# head = TECLinearHead()

# pooled = torch.load(
#     pooled_dir / "flare_00.pt",
#     map_location="cpu",
#     weights_only=True
# )

# prediction = head(pooled)

# print(prediction.shape)

In [ ]:
# train_idx, test_idx = next(loo.split(range(len(dataset))))

# head = TECLinearHead().to(device)

# optimizer = optim.Adam(
#     head.parameters(),
#     lr=1e-3
# )



In [90]:
latent_dir = Path("latent_features")
pooled_dir = Path("pooled_features")

In [99]:
epochs=40
fold_results = []

for fold, (train_idx,test_idx) in enumerate(loo.split(range(len(dataset))),start=1):
    head = TECLinearHead(low_res=(18, 36)).to(device)

    optimizer = optim.Adam(
        head.parameters(),
        lr=1e-3
    )

    head.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for i in train_idx:
            i = int(i)

            # Load pooled_features
            features = torch.load(
                Path("pooled_features") / f"flare_{i:02d}.pt",
                map_location=device,
                weights_only=True
            )

            

            # Load TEC Targets
            tec_path = dataset.df.iloc[i]["tec_grid_path"]

            tec, tec_mask = dataset.load_tec(tec_path)

            target = torch.from_numpy(tec).unsqueeze(0).to(device)
            mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

            optimizer.zero_grad()

            prediction = head(features)

            loss = masked_mse(prediction,target,mask)

            loss.backward()
            optimizer.step()

            total_loss+=loss.item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Training Loss: {total_loss/len(train_idx):.6f}"
        )

    with torch.no_grad():
        i = int(test_idx[0])

        features = torch.load(
            Path("pooled_features") / f"flare_{i:02d}.pt",
            map_location=device,
            weights_only=True
        )

        # Load TEC Targets
        tec_path = dataset.df.iloc[i]["tec_grid_path"]

        tec, tec_mask = dataset.load_tec(tec_path)

        target = torch.from_numpy(tec).unsqueeze(0).to(device)
        mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

        prediction = head(features)

        test_loss = masked_mse(prediction,target,mask)
        
    fold_results.append(test_loss.item())
    
    print(
        f"Fold {fold:02d}/{len(dataset)} | "
        f"Held-out MSE: {test_loss.item():.4f}\n"
    )

Epoch 1/40 | Training Loss: 427.753470
Epoch 2/40 | Training Loss: 342.818728
Epoch 3/40 | Training Loss: 284.881115
Epoch 4/40 | Training Loss: 243.355646
Epoch 5/40 | Training Loss: 212.660132
Epoch 6/40 | Training Loss: 189.328767
Epoch 7/40 | Training Loss: 171.189267
Epoch 8/40 | Training Loss: 156.826693
Epoch 9/40 | Training Loss: 145.280034
Epoch 10/40 | Training Loss: 135.873070
Epoch 11/40 | Training Loss: 128.116425
Epoch 12/40 | Training Loss: 121.648152
Epoch 13/40 | Training Loss: 116.196121
Epoch 14/40 | Training Loss: 111.553079
Epoch 15/40 | Training Loss: 107.559571
Epoch 16/40 | Training Loss: 104.091843
Epoch 17/40 | Training Loss: 101.053082
Epoch 18/40 | Training Loss: 98.366998
Epoch 19/40 | Training Loss: 95.973012
Epoch 20/40 | Training Loss: 93.822710
Epoch 21/40 | Training Loss: 91.877115
Epoch 22/40 | Training Loss: 90.104647
Epoch 23/40 | Training Loss: 88.479547
Epoch 24/40 | Training Loss: 86.980694
Epoch 25/40 | Training Loss: 85.590650
Epoch 26/40 | Tra

In [100]:
fold_results = np.array(fold_results)

print(f"Mean LOOCV MSE:   {fold_results.mean():.4f}")
print(f"Median LOOCV MSE: {np.median(fold_results):.4f}")
print(f"Std LOOCV MSE:    {fold_results.std():.4f}")
print(f"Min MSE:          {fold_results.min():.4f}")
print(f"Max MSE:          {fold_results.max():.4f}")

Mean LOOCV MSE:   96.9064
Median LOOCV MSE: 60.6287
Std LOOCV MSE:    81.3899
Min MSE:          34.8043
Max MSE:          396.0320


In [91]:
import torch.nn.functional as F

tokens = torch.load(
    latent_dir / "flare_00.pt",
    map_location="cpu",
    weights_only=True
)

# [B, L, D] -> [B, D, 256, 256]
spatial = tokens.transpose(1, 2).reshape(1, 1280, 256, 256)

# Divide spatial representation into 4x4 regions
pooled = F.adaptive_avg_pool2d(spatial, output_size=(4, 4))

# Flatten regional representations
pooled = pooled.flatten(start_dim=1)

print(pooled.shape)

torch.Size([1, 20480])


In [92]:
spatial_dir = Path("spatial_features")
spatial_dir.mkdir(exist_ok=True)

for i in range(len(dataset)):
    tokens = torch.load(
        latent_dir / f"flare_{i:02d}.pt",
        map_location="cpu",
        weights_only=True
    )

    spatial = tokens.transpose(1, 2).reshape(1, 1280, 256, 256)

    pooled = F.adaptive_avg_pool2d(
        spatial,
        output_size=(4, 4)
    ).flatten(start_dim=1)

    torch.save(pooled, spatial_dir / f"flare_{i:02d}.pt")

print("Spatial pooling complete.")

Spatial pooling complete.


In [97]:
epochs=20
fold_results = []

for fold, (train_idx,test_idx) in enumerate(loo.split(range(len(dataset))),start=1):
    head = TECLinearHead(
        input_dim=20480,
        low_res=(18, 36)
    ).to(device)

    optimizer = optim.Adam(
        head.parameters(),
        lr=1e-4
    )

    head.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for i in train_idx:
            i = int(i)

            # Load pooled_features
            tokens = torch.load(
                Path("spatial_features") / f"flare_{i:02d}.pt",
                map_location=device,
                weights_only=True
            )

            

            # Load TEC Targets
            tec_path = dataset.df.iloc[i]["tec_grid_path"]

            tec, tec_mask = dataset.load_tec(tec_path)

            target = torch.from_numpy(tec).unsqueeze(0).to(device)
            mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

            optimizer.zero_grad()

            prediction = head(tokens)

            loss = masked_mse(prediction,target,mask)

            loss.backward()
            optimizer.step()

            total_loss+=loss.item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Training Loss: {total_loss/len(train_idx):.6f}"
        )

    head.eval()
    with torch.no_grad():
        i = int(test_idx[0])

        tokens = torch.load(
            Path("spatial_features") / f"flare_{i:02d}.pt",
            map_location=device,
            weights_only=True
        )

        # Load TEC Targets
        tec_path = dataset.df.iloc[i]["tec_grid_path"]

        tec, tec_mask = dataset.load_tec(tec_path)

        target = torch.from_numpy(tec).unsqueeze(0).to(device)
        mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

        prediction = head(tokens)

        test_loss = masked_mse(prediction,target,mask)
        
    fold_results.append(test_loss.item())
    
    print(
        f"Fold {fold:02d}/{len(dataset)} | "
        f"Held-out MSE: {test_loss.item():.4f}\n"
    )

fold_results = np.array(fold_results)

print(f"Mean LOOCV MSE:   {fold_results.mean():.4f}")
print(f"Median LOOCV MSE: {np.median(fold_results):.4f}")
print(f"Std LOOCV MSE:    {fold_results.std():.4f}")
print(f"Min MSE:          {fold_results.min():.4f}")
print(f"Max MSE:          {fold_results.max():.4f}")

Epoch 1/20 | Training Loss: 353.805869
Epoch 2/20 | Training Loss: 219.835057
Epoch 3/20 | Training Loss: 166.654081
Epoch 4/20 | Training Loss: 138.164831
Epoch 5/20 | Training Loss: 120.684981
Epoch 6/20 | Training Loss: 108.868682
Epoch 7/20 | Training Loss: 100.212361
Epoch 8/20 | Training Loss: 93.476620
Epoch 9/20 | Training Loss: 88.007482
Epoch 10/20 | Training Loss: 83.432720
Epoch 11/20 | Training Loss: 79.521510
Epoch 12/20 | Training Loss: 76.119317
Epoch 13/20 | Training Loss: 73.116726
Epoch 14/20 | Training Loss: 70.433413
Epoch 15/20 | Training Loss: 68.009085
Epoch 16/20 | Training Loss: 65.797875
Epoch 17/20 | Training Loss: 63.764524
Epoch 18/20 | Training Loss: 61.881704
Epoch 19/20 | Training Loss: 60.128031
Epoch 20/20 | Training Loss: 58.486596
Fold 01/30 | Held-out MSE: 43.6722

Epoch 1/20 | Training Loss: 358.019189
Epoch 2/20 | Training Loss: 221.006234
Epoch 3/20 | Training Loss: 166.607163
Epoch 4/20 | Training Loss: 137.749316
Epoch 5/20 | Training Loss: 12

In [101]:
experiments = {
    "gap_18x36_20ep": {
        "feature_dir": "pooled_features",
        "input_dim": 1280,
        "low_res": (18, 36),
        "epochs": 20,
        "lr": 1e-3,
    },
    "gap_36x72_20ep": {
        "feature_dir": "pooled_features",
        "input_dim": 1280,
        "low_res": (36, 72),
        "epochs": 20,
        "lr": 1e-3,
    },
    "gap_45x90_20ep": {
        "feature_dir": "pooled_features",
        "input_dim": 1280,
        "low_res": (45, 90),
        "epochs": 20,
        "lr": 1e-3,
    },
    "spatial_4x4_20ep": {
        "feature_dir": "spatial_features",
        "input_dim": 20480,
        "low_res": (18, 36),
        "epochs": 20,
        "lr": 1e-4,
    },
    "gap_18x36_40ep": {
        "feature_dir": "pooled_features",
        "input_dim": 1280,
        "low_res": (18, 36),
        "epochs": 40,
        "lr": 1e-4,
    },
    "spatial_4x4_40ep": {
        "feature_dir": "spatial_features",
        "input_dim": 20480,
        "low_res": (18, 36),
        "epochs": 40,
        "lr": 1e-4,
    },
}

In [104]:
def run_loocv(config, dataset, device):
    loo = LeaveOneOut()
    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(
        loo.split(range(len(dataset))), start=1
    ):
        torch.manual_seed(42 + fold)
        head = TECLinearHead(
            input_dim=config["input_dim"],
            low_res=config["low_res"]
        ).to(device)

        optimizer = optim.Adam(
            head.parameters(),
            lr=config["lr"]
        )

        for epoch in range(config["epochs"]):
            head.train()

            for i in train_idx:
                i = int(i)

                tokens = torch.load(
                    Path(config["feature_dir"]) / f"flare_{i:02d}.pt",
                    map_location=device,
                    weights_only=True
                )

                tec_path = dataset.df.iloc[i]["tec_grid_path"]
                tec, tec_mask = dataset.load_tec(tec_path)

                target = torch.from_numpy(tec).unsqueeze(0).to(device)
                mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

                optimizer.zero_grad()

                prediction = head(tokens)
                loss = masked_mse(prediction, target, mask)

                loss.backward()
                optimizer.step()

        head.eval()

        with torch.no_grad():
            i = int(test_idx[0])

            tokens = torch.load(
                Path(config["feature_dir"]) / f"flare_{i:02d}.pt",
                map_location=device,
                weights_only=True
            )

            tec_path = dataset.df.iloc[i]["tec_grid_path"]
            tec, tec_mask = dataset.load_tec(tec_path)

            target = torch.from_numpy(tec).unsqueeze(0).to(device)
            mask = torch.from_numpy(tec_mask).unsqueeze(0).to(device)

            prediction = head(tokens)
            test_loss = masked_mse(prediction, target, mask)

        fold_results.append(test_loss.item())

        print(
            f"Fold {fold:02d}/{len(dataset)} | "
            f"Held-out MSE: {test_loss.item():.4f}"
        )

    return np.array(fold_results)

In [105]:
results = {}

for name, config in experiments.items():
    print(f"\nRunning experiment: {name}")

    fold_mse = run_loocv(config, dataset, device)

    results[name] = {
        "config": config,
        "fold_mse": fold_mse.tolist(),
        "mean": float(np.mean(fold_mse)),
        "median": float(np.median(fold_mse)),
        "std": float(np.std(fold_mse)),
        "min": float(np.min(fold_mse)),
        "max": float(np.max(fold_mse)),
    }


Running experiment: gap_18x36_20ep
Fold 01/30 | Held-out MSE: 38.0128
Fold 02/30 | Held-out MSE: 33.3309
Fold 03/30 | Held-out MSE: 49.7902
Fold 04/30 | Held-out MSE: 45.7549
Fold 05/30 | Held-out MSE: 41.1708
Fold 06/30 | Held-out MSE: 48.8436
Fold 07/30 | Held-out MSE: 53.0300
Fold 08/30 | Held-out MSE: 43.7290
Fold 09/30 | Held-out MSE: 57.3315
Fold 10/30 | Held-out MSE: 129.4893
Fold 11/30 | Held-out MSE: 37.9868
Fold 12/30 | Held-out MSE: 81.2315
Fold 13/30 | Held-out MSE: 99.1465
Fold 14/30 | Held-out MSE: 79.8506
Fold 15/30 | Held-out MSE: 200.9343
Fold 16/30 | Held-out MSE: 39.2898
Fold 17/30 | Held-out MSE: 155.1948
Fold 18/30 | Held-out MSE: 51.8677
Fold 19/30 | Held-out MSE: 52.0879
Fold 20/30 | Held-out MSE: 68.4320
Fold 21/30 | Held-out MSE: 51.9475
Fold 22/30 | Held-out MSE: 48.5748
Fold 23/30 | Held-out MSE: 108.2837
Fold 24/30 | Held-out MSE: 129.0858
Fold 25/30 | Held-out MSE: 118.4789
Fold 26/30 | Held-out MSE: 399.0089
Fold 27/30 | Held-out MSE: 182.5415
Fold 28/30 

In [106]:
summary = pd.DataFrame({
    name: {
        "Mean MSE": result["mean"],
        "Median MSE": result["median"],
        "Std MSE": result["std"],
        "Min MSE": result["min"],
        "Max MSE": result["max"],
    }
    for name, result in results.items()
}).T

display(summary.sort_values("Mean MSE"))

,Mean MSE,Median MSE,Std MSE,Min MSE,Max MSE
spatial_4x4_40ep,87.935655,56.315357,75.661854,32.665585,377.532959
spatial_4x4_20ep,93.386706,57.300814,81.232281,35.036640,404.597107
gap_18x36_20ep,110.025302,62.881750,105.230056,33.330902,470.792938
gap_36x72_20ep,111.872029,63.006355,108.146377,33.465042,476.892548
gap_45x90_20ep,112.856325,63.804604,109.324381,33.480999,479.623169
gap_18x36_40ep,239.017794,119.655525,241.595928,22.978977,959.790649


In [107]:
import json

with open("experiment_results.json", "w") as f:
    json.dump(results, f, indent=2)